# Attentify - scaling benchmark
This is separate from the ordinary tester. It compares **faces only**, **actions only**, **combined**, independent face/action input sizes, and output disabled versus smaller/larger preview encoding. It uses the same fixed weights. No training, minimum-hardware certification or attention accuracy is claimed.

Use the same clip on different machines to compare hardware. Use several genuine classroom clips with different student counts to investigate crowd scaling. **Detected face count is not true student count.** A larger inference size cannot restore details that were absent in the camera recording.

The default matrix has eight cases and two repeats: 16 trials per clip, processing at most the first 250 source frames each. It may take several minutes on GPU and much longer on CPU. Every case uses the same frame window and stride; even repeats reverse the case order to reduce ordering bias. Start with one short clip. Head pose is not integrated yet.


## 1. Load benchmark code and dependencies
Select a GPU runtime for CUDA. This notebook fetches a separate GitHub branch; it leaves the ordinary tester unchanged. Restart the runtime if switching ONNX Runtime package/backend after it has already been imported.

In [ ]:
import subprocess,sys,json
from pathlib import Path
CODE_REF='feature/scaling-benchmark' # Branch, tag or immutable commit SHA
FEATURE_MODE='combined' # 'faces', 'actions', 'combined'
BACKEND='cuda' # 'cpu' explicitly selects CPU action inference
if FEATURE_MODE not in ('faces','actions','combined'):raise ValueError('Choose faces, actions or combined')
dependencies=['opencv-python-headless==4.12.0.88','numpy==2.2.6']
if FEATURE_MODE!='faces':
    package='onnxruntime-gpu==1.23.2' if BACKEND=='cuda' else 'onnxruntime==1.23.2'
    # Both packages provide the same module; remove the alternate distribution.
    alternate='onnxruntime' if BACKEND=='cuda' else 'onnxruntime-gpu'
    subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',alternate],stdout=subprocess.DEVNULL)
    dependencies.append(package)
subprocess.check_call([sys.executable,'-m','pip','install','-q',*dependencies])
if FEATURE_MODE!='faces' and BACKEND=='cuda':
    import torch # Load Colab's CUDA/cuDNN libraries before ONNX Runtime
REPO=Path('/content/attentify-source')
if not REPO.exists():subprocess.check_call(['git','clone','https://github.com/NKD52/Attentify.git',str(REPO)])
dirty=subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
if dirty:raise RuntimeError('Source checkout has local changes. Save them before updating or use a new runtime.')
subprocess.check_call(['git','fetch','origin',CODE_REF],cwd=REPO)
subprocess.check_call(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO)
CODE=REPO/'computer-vision/colab'
sys.path[:0]=[str(CODE),str(REPO/'computer-vision/team-pilot')]
# Reload after fetching an update; existing engine objects are recreated in section 3.
for name in ['browser_camera','colab_runtime','workspace_storage','review_package','core','tracker','association']:
    sys.modules.pop(name,None)
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
if FEATURE_MODE!='faces' and BACKEND=='cuda':print('Remote GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
else:print('Selected inference uses CPU')

for module_name in ['benchmark','benchmark_resources']:
    sys.modules.pop(module_name,None)

subprocess.check_call([sys.executable,'-m','pip','install','-q','psutil>=6,<8'])


## 2. Private model storage
Set your own private model folder path. Drive model segments are joined and verified automatically; no repeated manual ZIP upload is needed. Connecting Drive grants the notebook access to it. Keep private paths and outputs out of GitHub commits.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODEL_SOURCE='/content/drive/MyDrive/YOUR_MODEL_FOLDER'
RESULTS_ROOT='/content/attentify-benchmarks' # Or a private writable Drive path
from workspace_storage import ensure_assets
assets=ensure_assets(MODEL_SOURCE,'/content/attentify-model-cache','combined')


## 3. Select recorded clips
Upload one or more clips, or provide paths already in Colab/Drive. Visible student counts are optional human-supplied metadata, not labels generated from face detection. Leave null/None if unknown. Files uploaded to Colab disappear when the runtime is deleted.

In [ ]:
INPUT_MODE='upload' # 'upload' or 'path'
CLIPS=[{'path':'/content/your-clip.mp4','label':'classroom-1','visible_students':None}]
if INPUT_MODE=='upload':
    from google.colab import files
    uploaded=files.upload()
    CLIPS=[{'path':name,'label':name,'visible_students':None} for name in uploaded]
    del uploaded
elif INPUT_MODE!='path':raise ValueError('Choose upload or path')
print(CLIPS)
# You can edit student counts in CLIPS before the next cell.


## 4. Run the controlled matrix
Sizes affect face detection and action inference independently. Output tests hold both model sizes at 640. Width 960 creates a small two-panel preview; width 3840 compares a larger encoding workload, not higher-resolution inference. The actual encoded dimensions are recorded. Preview movies retain the source frame rate: current raw source on the left, the matching dated inference snapshot on the right. Old boxes are never overlaid on current students.

GPU memory is a sampled maximum for the whole Python process, including initialization/warm-up and any retained context; it is not exact peak/model-exclusive VRAM. Missing measurements stay null. Review samples, if enabled, are generated after timing and contain identifiable images. Set REVIEW_EVERY=0 for speed-only runs. No outputs are published or sent automatically.

In [ ]:
from benchmark import make_cases,run_suite
ACTION_SIZES=(640,960) # Add 1280 later if needed; dynamic input sizes must be multiples of 32
FACE_WIDTHS=(640,960)
OUTPUT_WIDTHS=(960,3840) # Use (960,) to omit the more expensive large output case
REPEATS=2
MAX_SOURCE_FRAMES=250
INFERENCE_STRIDE=5
WARMUP_INFERENCES=3
REVIEW_EVERY=10 # One raw/numbered review sample per ten inference updates; 0 disables images
cases=make_cases(ACTION_SIZES,FACE_WIDTHS,OUTPUT_WIDTHS)
print('Trials:',len(CLIPS)*len(cases)*REPEATS)
result_zip=run_suite(CLIPS,assets,cases=cases,backend=BACKEND,results_root=RESULTS_ROOT,
    stride=INFERENCE_STRIDE,max_source_frames=MAX_SOURCE_FRAMES,repeats=REPEATS,
    warmup=WARMUP_INFERENCES,review_every=REVIEW_EVERY)
print('Saved:',result_zip)


## 5. Inspect and download results
Compare decode, face processing, action preprocessing/runtime/postprocessing, rendering and encoding separately. Pipeline throughput includes decoding every source frame, including skipped inference frames, and encoder finalization. Model initialization and warm-up are reported separately. The timed pipeline excludes final review generation, JSON/CSV export, ZIP creation and upload/download.

**This is offline:** source-time snapshot gaps describe the scheduling gap in the output, not camera exposure-to-screen latency. Failed cases remain failed and are not silently retried with smaller inputs or CPU. Compare quality using the exported reviews before choosing a faster configuration. A test with few inference updates is only a smoke test.

In [ ]:
import pandas as pd,zipfile,json
with zipfile.ZipFile(result_zip) as z:
    name=next(n for n in z.namelist() if n.endswith('comparison.csv'))
    with z.open(name) as f:table=pd.read_csv(f)
columns=['source_label','repeat','case_name','status','median_model_ms','pipeline_elapsed_s',
         'decoded_source_frames_per_wall_s','inference_updates_per_wall_s','realtime_factor',
         'median_yunet_tracking_ms','median_action_runtime_ms',
         'mean_detected_faces','sampled_max_process_gpu_memory_mib']
display(table[[c for c in columns if c in table]])
from google.colab import files
files.download(str(result_zip))


Extract the ZIP. `comparison.csv` and `comparison.json` summarize all cases; each case folder contains settings, `frames.csv`, predictions and optional preview/review files. Open a case's `review/index.html` and export answers before switching browsers. Send the run ZIP plus exported review JSONs for analysis. Compare identical sources/code/model versions and repeated trials; do not infer minimum hardware or classroom accuracy from a single clip.

Next: collect genuine clips with larger classes, review resolution/quality trade-offs, and only then choose an optimization. Smaller models, lower precision, temporal scheduling and head-pose batching remain future experiments, not changes silently applied here.
